In [ ]:
!pip install -q langchain langchain-community langchain-google-genai langchain-text-splitters pymupdf faiss-cpu

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import PromptTemplate
from langchain_google_genai import (
    GoogleGenerativeAIEmbeddings,
    ChatGoogleGenerativeAI
)

import os


In [ ]:
# Step 1: Load Documents
# loader = PyMuPDFLoader("/application/deepanshu_sde(1).pdf")
loader = PyMuPDFLoader("/content/deepanshu_sde (1).pdf")
docs = loader.load()

In [ ]:
# Step 2: Split Documents
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
split_documents = text_splitter.split_documents(docs)


In [ ]:
# Step 3: Generate Embeddings
from getpass import getpass
os.environ["GOOGLE_API_KEY"] = getpass("Enter API Key: ")

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"   # was "models/embedding-001"
)

test = embeddings.embed_query("hello")
print(len(test))   # should print 3072 by default



In [ ]:
# Step 4: Create and Save the Database
vectorstore = FAISS.from_documents(documents=split_documents, embedding=embeddings)


In [ ]:
# Step 5: Create Retriever
retriever = vectorstore.as_retriever()


In [ ]:
# Step 6: Create Prompt
prompt = PromptTemplate.from_template(
    """You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.

#Context:
{context}

#Question:
{question}

#Answer:"""
)



In [ ]:
# Step 7: Load LLM
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0)  # `model=`, not `model_name=`

In [ ]:
# Step 8: Create Chain
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)


In [ ]:
# Try it
answer = chain.invoke("What is this document about?")
print(answer)